# Practice 2. Clustering and informative features

Industrial AI, KBTU. Goes with Lecture 3 and `inclass_work_3.ipynb`. **1.25 points.**

**Data: the Tennessee Eastman process** (TEP), a simulated chemical plant that has been the
standard benchmark for fault detection since 1993. 52 variables, six operating modes: normal
operation and five faults. The loader cell downloads the files from Kaggle
(`red2yang/tennessee-eastman-profbraatz`) once.

The `fault` column is there so that you can check your clusters. Your clustering must never use
it as an input. Use it only in crosstabs, in ARI, and for mutual information in Tasks 6–7.

**The process.** Four gaseous reactants A, C, D, E and an inert B go into a reactor and make two
liquid products G and H, plus a by-product F. After the reactor come a condenser, a
vapour-liquid separator, a recycle compressor and a stripper. The simulation is closed-loop: a
control system holds pressures, levels and temperatures at their set points.

**What a row is.** One sample every 3 minutes. Each training file holds 24 hours of operation,
480 samples.

**The variables you will meet most** (the full list of 52 is in Downs & Vogel, 1993):

| tag | what it is | tag | what it is |
|---|---|---|---|
| `XMEAS1` | A feed, stream 1 | `XMEAS18` | stripper temperature |
| `XMEAS4` | A and C feed, stream 4 | `XMEAS19` | stripper steam flow |
| `XMEAS7` | reactor pressure | `XMEAS21` | reactor cooling water outlet temperature |
| `XMEAS9` | reactor temperature | `XMEAS23`–`XMEAS36` | gas analysers, reactor feed and purge, sampled every 6 min |
| `XMEAS10` | purge rate | `XMEAS37`–`XMEAS41` | product analyser, sampled every 15 min |
| `XMV3` | A feed flow valve | `XMV9` | stripper steam valve |
| `XMV4` | A and C feed flow valve | `XMV10` | reactor cooling water flow valve |
| `XMV6` | purge valve | `XMV11` | condenser cooling water flow valve |

**The six modes in this practice**, the `fault` column:

| fault | what happens |
|---|---|
| 0 | normal operation |
| 1 | step in the A/C feed ratio, B composition constant (stream 4) |
| 2 | step in the B composition, A/C ratio constant (stream 4) |
| 4 | step in the reactor cooling water inlet temperature |
| 6 | loss of the A feed (stream 1) |
| 7 | loss of C header pressure, reduced availability (stream 4) |

### How the tasks are built

Each task says what to write: the functions, their arguments and what they return, and sometimes
a variable with a result. How you write them is up to you, as long as you use the method the task
names. Under your cell sits a cell of **tests**. Run it after your code: it either prints
`all tests passed` or stops at the first check that failed and says what is wrong. The tests check
what your code does on small cases with known answers, not how it is written.

These tests are public. When you hand in, the same tests run again together with hidden ones, on
other inputs, and the share of tests passed gives the points for the code. Code that only fits the
public cases loses those points.

The cell marked *Common ground* is written for you: it standardises the 52 variables into `Z`
and takes the labels into `y`, the same for everybody.

The methods are yours: the run length, Lloyd's K-means, the silhouette, DBSCAN's core points, the
adjusted Rand index, entropy and binned mutual information, scaling against normal operation.
Library versions of these are for checking your result only. Where a task says so, the results
on the full data come from scikit-learn (`KMeans`, `silhouette_score`, `DBSCAN`,
`mutual_info_classif`): the by-hand versions are written for understanding, and some of them
would need gigabytes on 2 900 samples. Distances may come from `scipy.spatial.distance.cdist`.

### What you hand in

Two files:

1. this notebook, run top to bottom, named `Surname_Name_practice_2.ipynb`;
2. `practice_2_answers.md` with your answers to the questions under each task, named
   `Surname_Name_practice_2_answers.md`.

Every answer needs the numbers your notebook printed. A number the notebook does not print scores
zero, and so does a number printed with nothing said about it.

| Task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | total |
|---|---|---|---|---|---|---|---|---|
| code (tests) | 0.04 | 0.10 | 0.07 | 0.10 | 0.07 | 0.10 | 0.10 | 0.58 |
| answers | 0.06 | 0.10 | 0.08 | 0.10 | 0.08 | 0.10 | 0.15 | 0.67 |

### The rules

- The fault label is never an input to a clustering. Using it there scores zero for the task.
- Every choice of $k$, $\varepsilon$ or a feature needs a reason written next to it.
- Do not rename the functions and variables the tests use, and do not edit the test cells.
- Before handing in, restart the kernel and run all cells; the whole notebook runs in about two
  minutes on a laptop. Then check your answers against the new output.
- Figures need axis labels and a title that says what to look at.
- Stuck? `inclass_work_3.ipynb` does every step on the hydraulic rig.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import sklearn

BLUE, ORANGE, RED, GREEN, PURPLE, GREY = (
    "#2a6f8e", "#e07b39", "#c0392b", "#4a8f5a", "#9b3f9b", "#9aa0a6")

mpl.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 10,
    "font.size": 10,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

KAGGLE = {
    'tep': 'red2yang/tennessee-eastman-profbraatz',
}


def dataset_folder(name):
    """The folder with the files of a data set: the Kaggle copy (downloaded once, then cached),
    or data/<name>/ of the course repo if Kaggle is out of reach."""
    if KAGGLE[name]:
        try:
            import contextlib, io, logging, kagglehub
            logging.getLogger("kagglehub").setLevel(logging.ERROR)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                folder = Path(kagglehub.dataset_download(KAGGLE[name]))
            print(f"{name}: Kaggle {KAGGLE[name]}")
            return folder
        except Exception as exc:
            print(f"Kaggle unavailable ({type(exc).__name__}), looking for data/{name}/")
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / name).is_dir():
            return root / "data" / name
    raise FileNotFoundError(f"no data/{name}/ here or above; run data/fetch.py in the course repo")

print(" | ".join(f"{m} {sys.modules[m].__version__}" for m in
                 ("numpy", "pandas", "matplotlib", "scipy", "sklearn", "torch") if m in sys.modules))

numpy 2.5.5 | pandas 3.1.1 | matplotlib 3.11.2 | scipy 1.18.1 | sklearn 1.9.1


In [2]:
FAULTS = [0, 1, 2, 4, 6, 7]
XMEAS = [f"XMEAS{i}" for i in range(1, 42)]      # 41 measured variables
XMV = [f"XMV{i}" for i in range(1, 12)]           # 11 manipulated variables (valves)
VARS = XMEAS + XMV


def load_tep():
    # training files d00.dat ... d21.dat; we take the six modes in FAULTS
    folder = next(dataset_folder("tep").rglob("d00.dat")).parent
    parts = []
    for f in FAULTS:
        a = np.loadtxt(folder / f"d{f:02d}.dat")
        print(f"  d{f:02d}.dat  as stored: {a.shape}")
        if a.shape[0] == len(VARS):          # one of the files is stored on its side
            a = a.T
        part = pd.DataFrame(a, columns=VARS)
        parts.append(part.assign(fault=f, sample=np.arange(len(a))))
    return pd.concat(parts, ignore_index=True)


tep = load_tep()
print(tep.shape)

tep: Kaggle red2yang/tennessee-eastman-profbraatz
  d00.dat  as stored: (52, 500)
  d01.dat  as stored: (480, 52)
  d02.dat  as stored: (480, 52)
  d04.dat  as stored: (480, 52)
  d06.dat  as stored: (480, 52)
  d07.dat  as stored: (480, 52)
(2900, 54)


*Common ground: the standardised variables and the labels, the same for everybody. Run it as it is.*

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score

Z = StandardScaler().fit_transform(tep[VARS])  # 2900 x 52, mean 0, std 1 per variable
y = tep["fault"].to_numpy()             # for checking only, never a clustering input

modes, counts = np.unique(y, return_counts=True)
print(Z.shape, "| modes:", dict(zip(modes, counts)))

(2900, 52) | modes: {np.int64(0): np.int64(500), np.int64(1): np.int64(480), np.int64(2): np.int64(480), np.int64(4): np.int64(480), np.int64(6): np.int64(480), np.int64(7): np.int64(480)}


---

## Task 1. What is in the file  
*0.10 points: 0.04 for the code (tests), 0.06 for the answers*

Take inventory before you cluster anything. A historian logs every variable on one clock, but an
analyser that reports every 15 minutes leaves its last value in the table until the next report.
Such a variable is **held**: it repeats the same value on several consecutive samples. The mean
run length tells you how long a value is held. With runs of lengths $\ell_1, \dots, \ell_r$ of
identical consecutive values in a series of $n$ samples,

$$\bar\ell = \frac1r \sum_j \ell_j = \frac{n}{r}.$$

A variable that changes every sample has $\bar\ell = 1$; one reported every fifth sample has
$\bar\ell \approx 5$.

**Write**

- `mean_run_length(s)`: the mean length of the runs of identical consecutive values of a
  `pd.Series` without missing values, as a `float`;
- `n_distinct`: a `pd.Series` indexed by `VARS`, the number of distinct values of each variable
  in `tep`;
- `held`: a `pd.Series` indexed by `VARS`, the mean run length of each variable in the rows of
  normal operation (`fault == 0`), in their order.

In [ ]:
def mean_run_length(s):
    ...


n_distinct = ...
held = ...

In [ ]:
# Tests for Task 1 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_three_runs():
    got = mean_run_length(pd.Series([1, 1, 2, 2, 2, 3]))
    assert isinstance(got, float), f"return a float, got {type(got).__name__}"
    assert np.isclose(got, 2.0), f"runs of 2, 3 and 1: mean 2.0, got {got}"


test_three_runs()


def test_no_repeats_and_all_repeats():
    all_differ = mean_run_length(pd.Series([3.0, 1.0, 4.0, 1.5]))
    one_run = mean_run_length(pd.Series([7.0] * 6))
    broken_runs = mean_run_length(pd.Series([1, 2, 1, 2, 2]))
    assert np.isclose(all_differ, 1.0), "every value differs: 1.0"
    assert np.isclose(one_run, 6.0), "one run of 6"
    assert np.isclose(broken_runs, 1.25), "1 does not join a later 1: runs 1, 1, 1, 2"


test_no_repeats_and_all_repeats()


def test_results_have_the_right_type():
    assert isinstance(n_distinct, pd.Series), "n_distinct: a Series indexed by VARS"
    assert list(n_distinct.index) == VARS, "n_distinct: a Series indexed by VARS"
    assert isinstance(held, pd.Series), "held: a Series indexed by VARS"
    assert list(held.index) == VARS, "held: a Series indexed by VARS"


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_2_answers.md`, Task 1, with the numbers your code printed):

1. How many rows does each mode have, and how many hours does a file cover? Why did `d00.dat` need the special line in the loader?
2. Which variables have far fewer distinct values than the others? Use `held` to say how long their values are held, and explain why.
3. Report the range of `XMV10` in each mode. What is odd about fault 6, and from which hour of the file?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 2. K-means and the elbow  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

K-means minimises the inertia
$J = \sum_i \lVert \mathbf{x}_i - \boldsymbol{\mu}_{c(i)} \rVert^2$, where $c(i)$ is the cluster
of point $i$. Lloyd's algorithm alternates two steps until the centres stop moving:

1. **assign**: $c(i) = \arg\min_j \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert$;
2. **update**: $\boldsymbol{\mu}_j = \dfrac{1}{|C_j|} \sum_{i \in C_j} \mathbf{x}_i$.

The first step is the best assignment for fixed centres, the second the best centres for a fixed
assignment (the mean minimises the sum of squared distances), so $J$ never grows. It stops in a
local minimum that depends on the start, which is why `KMeans` restarts from several starts
(`n_init`) and keeps the best.

**Write**

- `kmeans_lloyd(X, init_centres, n_iter=300)`: Lloyd's algorithm from the given $k \times p$
  starting centres, at most `n_iter` rounds, stopping as soon as the centres no longer change.
  Returns `(labels, centres, inertia)`: an integer array of length $n$ with values $0 \dots k-1$
  (ties go to the lower index, as `argmin` does), the $k \times p$ final centres, and $J$ for the
  final centres and the labels nearest to them, as a `float`. A cluster that loses all its points
  keeps its centre where it was;
- on the full data, with `KMeans(k, n_init=10, random_state=0).fit(Z)` for $k = 2 \dots 10$:
  `km_labels`, a dict $k \to$ the array of labels, `km_inertia`, a dict $k \to$ the inertia, and
  `ari_k6`, the adjusted Rand index of the $k = 6$ partition against `y`
  (`adjusted_rand_score`).

In [ ]:
def kmeans_lloyd(X, init_centres, n_iter=300):
    ...


km_labels = ...
km_inertia = ...
ari_k6 = ...

In [ ]:
# Tests for Task 2 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_four_points_on_a_line():
    X = np.array([[0.0], [1.0], [10.0], [11.0]])
    labels, centres, J = kmeans_lloyd(X, np.array([[0.0], [10.0]]))
    assert list(labels) == [0, 0, 1, 1], f"labels {list(labels)}"
    message = f"centres {centres.ravel()}, inertia {J}"
    assert np.allclose(centres, [[0.5], [10.5]]) and np.isclose(J, 1.0), message


test_four_points_on_a_line()


def test_same_start_same_result_as_sklearn():
    from sklearn.datasets import make_blobs
    Xd, _ = make_blobs(600, centers=4, cluster_std=1.5, random_state=1)
    init = Xd[[0, 50, 100, 150]]
    lab, cen, J = kmeans_lloyd(Xd, init)
    ref = KMeans(4, init=init, n_init=1, algorithm="lloyd", tol=0).fit(Xd)
    assert np.isclose(J, ref.inertia_), f"inertia {J:.2f}, sklearn {ref.inertia_:.2f}"
    assert (np.asarray(lab) == ref.labels_).all(), \
        "the labels differ from sklearn's with the same start"


test_same_start_same_result_as_sklearn()


def test_results_have_the_right_type():
    ks = list(range(2, 11))
    assert isinstance(km_labels, dict) and sorted(km_labels) == ks, "km_labels: keys 2..10"
    assert isinstance(km_inertia, dict) and sorted(km_inertia) == ks, "km_inertia: keys 2..10"
    assert all(len(km_labels[k]) == len(Z) for k in km_labels), "one label per sample"
    assert isinstance(ari_k6, float)


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_2_answers.md`, Task 2, with the numbers your code printed):

1. Report the largest and the smallest raw standard deviation among the 52 variables. Why can standardisation not be skipped here?
2. Plot `km_inertia` against $k$ and report the relative drop from $k-1$ to $k$. Is there an elbow?
3. Crosstab the $k = 6$ clusters against `fault` and report `ari_k6`. Which faults did K-means find, which did it merge, and with what?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 3. The silhouette  
*0.15 points: 0.07 for the code (tests), 0.08 for the answers*

For point $i$ in cluster $C$:

- $a(i)$: the mean distance from $i$ to the **other** points of $C$;
- $b(i)$: for every other cluster $C'$, the mean distance from $i$ to the points of $C'$; take
  the **smallest** of these;

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i),\ b(i)\}} \in [-1, 1].$$

$s(i)$ near 1: the point is much closer to its own cluster than to the next one. Near 0: it sits
on the border. Negative: it would be better off in the neighbouring cluster. A point alone in its
cluster has no $a(i)$; by convention $s(i) = 0$ (as in scikit-learn). The mean of $s(i)$ over all
points scores a whole partition, and unlike the inertia it does not improve just because $k$
grows.

**Write**

- `silhouette_by_hand(X, labels)`: the array of $s(i)$, one per point, for at least two clusters;
  `labels` may be any integers, not necessarily $0 \dots k-1$;
- `sil`: a dict $k \to$ `silhouette_score(Z, km_labels[k])` for $k = 2 \dots 10$. Your by-hand
  version needs all $n^2$ distances, so it is checked on small data.

In [ ]:
def silhouette_by_hand(X, labels):
    ...


sil = ...

In [ ]:
# Tests for Task 3 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_matches_sklearn_on_blobs():
    from sklearn.datasets import make_blobs
    from sklearn.metrics import silhouette_samples
    Xs, _ = make_blobs(300, centers=3, cluster_std=2.0, random_state=7)
    lab_s = KMeans(3, n_init=10, random_state=0).fit_predict(Xs)
    mine = silhouette_by_hand(Xs, lab_s)
    ref = silhouette_samples(Xs, lab_s)
    assert np.allclose(mine, ref), "s(i) differs from silhouette_samples"


test_matches_sklearn_on_blobs()


def test_four_points_by_hand():
    X = np.array([[0.0], [2.0], [10.0], [12.0]])
    s = silhouette_by_hand(X, np.array([0, 0, 1, 1]))
    # point 0: a = 2, b = (10 + 12) / 2 = 11, s = 9 / 11
    assert np.isclose(s[0], 9 / 11) and np.isclose(s[1], 7 / 9), f"got {np.round(s, 4)}"


test_four_points_by_hand()


def test_a_point_alone_scores_zero():
    X = np.array([[0.0], [1.0], [5.0]])
    s = silhouette_by_hand(X, np.array([4, 4, 9]))
    assert np.isclose(s[2], 0.0), "a cluster of one point: s = 0"
    assert np.isclose(s[0], (5 - 1) / 5), "labels need not be 0..k-1"


test_a_point_alone_scores_zero()


def test_sil_has_every_k():
    assert isinstance(sil, dict) and sorted(sil) == list(range(2, 11)), "sil: keys 2..10"
    assert all(-1 <= v <= 1 for v in sil.values())


test_sil_has_every_k()


print("all tests passed")

**Questions** (answer them in `practice_2_answers.md`, Task 3, with the numbers your code printed):

1. Plot `sil` against $k$ next to the ARI of each partition against `fault`. Which $k$ does the silhouette choose?
2. Crosstab the best-silhouette partition against `fault`. What did it actually separate?
3. The elbow, the silhouette and the true number of modes do not agree. Explain why a high silhouette can go with a partition that is useless to an engineer.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 4. DBSCAN on the same data  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

A point is a **core point** if at least `min_samples` points, **itself included**, lie within
distance $\varepsilon$ of it ($\lVert \mathbf{x}_j - \mathbf{x}_i \rVert \le \varepsilon$). A
cluster is a maximal set of core points that reach each other through chains of neighbours,
plus the border points within $\varepsilon$ of them; every other point is noise (label `-1`).
DBSCAN needs no $k$, but $\varepsilon$ sets the density a cluster must have.

**Choosing $\varepsilon$.** For every point take the distance to its `min_samples`-th nearest
neighbour, counting the point itself as the first. Sorted, these distances form the
**k-distance curve**: points inside dense regions sit at the bottom, noise at the top, and a
sharp bend, if there is one, marks a good $\varepsilon$.

**Write**

- `core_mask(X, eps, min_samples)`: a boolean array, `True` for the core points;
- `kdist`: the k-distance curve of `Z` for `min_samples = 10`: an array of length 2 900 in
  ascending order (`NearestNeighbors(n_neighbors=10)` counts the point itself);
- `eps_sweep`: a `pd.DataFrame` indexed by $\varepsilon$ = 3.0, 3.5, 4.0, 4.5, 5.0, 6.0 with the
  columns `"clusters"` (not counting noise), `"noise"` (number of noise points) and `"ARI"`
  (against `y`), for `DBSCAN(eps, min_samples=10)` on `Z`;
- `EPS`: your choice of $\varepsilon$, a `float`, argued from the k-distance curve and not from
  ARI; `db_labels`: `DBSCAN(EPS, min_samples=10).fit_predict(Z)`.

In [ ]:
def core_mask(X, eps, min_samples):
    ...


kdist = ...
eps_sweep = ...
EPS = ...
db_labels = ...

In [ ]:
# Tests for Task 4 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_four_points_on_a_line():
    X = np.array([[0.0], [1.0], [2.0], [10.0]])
    got = core_mask(X, eps=1.0, min_samples=2)
    assert isinstance(got, np.ndarray) and got.dtype == bool, "a boolean array"
    assert got.tolist() == [True, True, True, False], \
        "distance exactly eps counts; the point itself counts"
    assert core_mask(X, eps=1.0, min_samples=3).tolist() == [False, True, False, False]


test_four_points_on_a_line()


def test_same_core_points_as_sklearn():
    from sklearn.datasets import make_moons
    Xm, _ = make_moons(400, noise=0.08, random_state=0)
    ref = np.zeros(len(Xm), dtype=bool)
    core_points = DBSCAN(eps=0.15, min_samples=6).fit(Xm).core_sample_indices_
    ref[core_points] = True
    mine = core_mask(Xm, eps=0.15, min_samples=6)
    assert (np.asarray(mine) == ref).all(), f"core points {np.sum(mine)}, sklearn {ref.sum()}"


test_same_core_points_as_sklearn()


def test_results_have_the_right_type():
    message = "kdist: one distance per sample, ascending"
    assert isinstance(kdist, np.ndarray) and kdist.shape == (len(Z),), message
    assert np.all(np.diff(kdist) >= 0), message
    assert isinstance(eps_sweep, pd.DataFrame)
    assert list(eps_sweep.index) == [3.0, 3.5, 4.0, 4.5, 5.0, 6.0]
    assert {"clusters", "noise", "ARI"} <= set(eps_sweep.columns)
    assert isinstance(EPS, float) and len(db_labels) == len(Z)


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_2_answers.md`, Task 4, with the numbers your code printed):

1. Draw the k-distance plot with your $\varepsilon$ marked, and report a few quantiles of `kdist`. Why did you choose this $\varepsilon$?
2. Report `eps_sweep`. What happens to the noise and the clusters as $\varepsilon$ grows?
3. Crosstab `db_labels` against `fault`. Which modes end up as noise, which share a cluster?
4. Why does one $\varepsilon$ not fit this data? Compare how spread out each mode is: the mean over the 52 columns of `Z[y == f].std(axis=0)`.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 5. Comparing the two partitions  
*0.15 points: 0.07 for the code (tests), 0.08 for the answers*

The adjusted Rand index from the contingency table $n_{ij}$ of two partitions (row sums $a_i$,
column sums $b_j$, $n$ points in total):

$$\mathrm{ARI} = \frac{\sum_{ij}\binom{n_{ij}}{2} - E}{\tfrac12\left[\sum_i\binom{a_i}{2} + \sum_j\binom{b_j}{2}\right] - E},
\qquad E = \frac{\sum_i\binom{a_i}{2}\,\sum_j\binom{b_j}{2}}{\binom{n}{2}}.$$

$\binom{m}{2} = m(m-1)/2$ is the number of pairs among $m$ points. $\sum_{ij}\binom{n_{ij}}{2}$
counts the pairs that both partitions put together. $E$ is what that count would be on average
if the two partitions were independent with the same group sizes, and the denominator is its
largest possible value minus $E$. So ARI is 0 for chance agreement, 1 for identical partitions
whatever the names of the groups, and can go below 0. It is symmetric, and it never needs to know
which cluster "is" which fault.

**Write**

- `ari_by_hand(a, b)`: the ARI of two label sequences of the same length, as a `float`; labels may
  be any hashable values;
- `ari_table`: a dict with your ARI of `km_labels[6]` against `y` (`"kmeans_fault"`), of
  `db_labels` against `y` (`"dbscan_fault"`) and of `km_labels[6]` against `db_labels`
  (`"kmeans_dbscan"`);
- `shift`: a `pd.DataFrame` indexed by the faults 1, 2, 4, 6, 7 with the columns `VARS`: how far the
  mean of each variable in that fault lies from its mean in normal operation, in standard
  deviations of normal operation (`std` with $n - 1$).

In [ ]:
def ari_by_hand(a, b):
    ...


ari_table = {"kmeans_fault": ..., "dbscan_fault": ..., "kmeans_dbscan": ...}
shift = ...

In [ ]:
# Tests for Task 5 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_agreement_and_anti_agreement():
    got = ari_by_hand([0, 0, 1, 1], [5, 5, 7, 7])
    assert isinstance(got, float) and np.isclose(got, 1.0), \
        f"the same partition under other names: 1.0, got {got}"
    assert np.isclose(ari_by_hand([0, 0, 1, 1], [0, 1, 0, 1]), -0.5), \
        "no pair kept together: (0 - 2/3) / (2 - 2/3) = -0.5"


test_agreement_and_anti_agreement()


def test_matches_sklearn_on_a_noisy_copy():
    rng = np.random.default_rng(1)
    a = rng.integers(0, 4, 1000)
    b = np.where(rng.random(1000) < 0.7, a, rng.integers(0, 5, 1000))
    assert np.isclose(ari_by_hand(a, b), adjusted_rand_score(a, b)), \
        "differs from adjusted_rand_score"


test_matches_sklearn_on_a_noisy_copy()


def test_results_have_the_right_type():
    assert set(ari_table) == {"kmeans_fault", "dbscan_fault", "kmeans_dbscan"}
    assert all(isinstance(v, float) for v in ari_table.values()), "three floats"
    assert isinstance(shift, pd.DataFrame)
    assert list(shift.index) == [1, 2, 4, 6, 7] and list(shift.columns) == VARS


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_2_answers.md`, Task 5, with the numbers your code printed):

1. Crosstab K-means ($k = 6$) against your DBSCAN and report `ari_table`. Where do the two methods disagree most?
2. Which modes do **both** methods separate, which does **neither**?
3. For the modes neither method finds, use `shift`: which variables move, and by how much? Why does a fault visible in one variable hide in a 52-dimensional distance?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 6. Which variables carry information  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

Three filters rank variables without a model: variance, correlation and mutual information.

**Mutual information** as a reduction of uncertainty, in nats (natural logarithm):

$$H(Y) = -\sum_y p(y)\ln p(y), \qquad
H(Y \mid B) = \sum_b p(b)\, H(Y \mid B=b), \qquad
I(B;Y) = H(Y) - H(Y \mid B).$$

$B$ is the variable cut into `bins` bins with equal counts. $H(Y \mid B)$ is the entropy of the
labels inside each bin, weighted by the share of rows in the bin: what is left uncertain once you
know which bin a sample fell into. $0 \le I \le H(Y)$; $I = H(Y)$ means the bin tells the label
exactly. Cut the bins exactly like this, so that ties are broken the same way for everybody:

```python
b = pd.qcut(pd.Series(np.asarray(x)).rank(method="first"), bins, labels=False).to_numpy()
```

`mutual_info_classif` estimates the same quantity differently: from the distances to the nearest
neighbours of each sample, without bins.

**Write**

- `entropy(labels)`: $H$ of a sequence of labels, as a `float`;
- `binned_mi(x, y, bins=10)`: $I(B;Y)$ for a numeric sequence `x` and labels `y`, as a `float`;
- `mm_var`: a `pd.Series` indexed by variable, the variance (population, numpy default) of each
  of the 52 variables of `tep` after min-max scaling to $[0, 1]$, smallest first;
- `n_high_corr`: an `int`, the number of pairs of different variables with $|r| > 0.95$;
- `mi_knn`, `mi_bin`, `mi_knn5`: `pd.Series` indexed by variable, largest first: the MI of each
  variable with `y` by `mutual_info_classif(tep[VARS], y, random_state=0)`, by your `binned_mi`,
  and by `mutual_info_classif` on every fifth row only (`tep.iloc[::5]`, `random_state=0`).

In [ ]:
def entropy(labels):
    ...


def binned_mi(x, y, bins=10):
    ...


mm_var = ...
n_high_corr = ...
mi_knn = ...
mi_bin = ...
mi_knn5 = ...

In [ ]:
# Tests for Task 6 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_entropy_known_values():
    got = entropy([0, 1, 2, 3] * 5)
    assert isinstance(got, float) and np.isclose(got, np.log(4)), \
        f"four equal classes: ln 4 = 1.386, got {got}"
    assert np.isclose(entropy(["a"] * 7), 0.0), "one class: 0"
    assert np.isclose(entropy([0, 0, 0, 1]), -(0.75 * np.log(0.75) + 0.25 * np.log(0.25)))


test_entropy_known_values()


def test_binned_mi_matches_the_plugin_estimate():
    from sklearn.metrics import mutual_info_score
    rng = np.random.default_rng(0)
    cls = rng.integers(0, 3, 3000)
    feat = cls + rng.normal(0, 0.8, 3000)
    bins = pd.qcut(pd.Series(feat).rank(method="first"), 10, labels=False)
    got = binned_mi(feat, cls)
    assert isinstance(got, float) and np.isclose(got, mutual_info_score(cls, bins)), \
        "differs from mutual_info_score on the same bins"


test_binned_mi_matches_the_plugin_estimate()


def test_a_perfect_feature_carries_all_of_h():
    lab = np.repeat([0, 1], 50)
    feature = lab + 0.01 * np.arange(100)
    assert np.isclose(binned_mi(feature, lab, bins=2), np.log(2)), \
        "two bins that split the classes: I = H = ln 2"


test_a_perfect_feature_carries_all_of_h()


def test_results_have_the_right_type():
    results = {"mm_var": mm_var, "mi_knn": mi_knn, "mi_bin": mi_bin, "mi_knn5": mi_knn5}
    for name, s in results.items():
        assert isinstance(s, pd.Series) and set(s.index) == set(VARS), \
            f"{name}: a Series indexed by the 52 variables"
    assert mm_var.is_monotonic_increasing and mi_bin.is_monotonic_decreasing, \
        "mm_var smallest first, MI largest first"
    assert isinstance(n_high_corr, (int, np.integer))


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_2_answers.md`, Task 6, with the numbers your code printed):

1. List the ten smallest raw variances and the ten smallest of `mm_var`. Is `XMEAS9`, the reactor temperature, a dead sensor? Use Task 1 and one sentence about what a controller does to the variable it controls.
2. Report `n_high_corr` and the strongest pairs. Say what physical pair each of five of them is.
3. Print the top ten of `mi_knn`, `mi_bin` and `mi_knn5` side by side, with $H(Y)$. The two estimators disagree about a whole group of variables. Which group, why is one estimator fooled, and how does `mi_knn5` confirm it?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 7. A justified short list of five  
*0.25 points: 0.10 for the code (tests), 0.15 for the answers*

A ranking by mutual information with all six modes favours the variables that separate the
*big* faults. Ask each fault separately: on the rows of fault $f$ and of normal operation only,
the binned MI of each variable with the label. Its ceiling is $\ln 2 = 0.69$.

**The short list** is five variables that

- **cover every fault**: for each fault $f$, some chosen variable has a per-fault MI of at least
  half the best per-fault MI of that fault;
- **do not repeat each other**: no two chosen variables have $|r| > 0.95$ on `tep`.

**Scaling against normal operation.** `StandardScaler` divides by the standard deviation over
all data, and a fault that drives a valve to its limit inflates that number for everybody. Process
monitoring scales against normal operation instead: with $\bar x_0$ and $s_0$ the mean and the
standard deviation ($n - 1$) of each column over the normal rows,

$$z = \mathrm{clip}\!\left(\frac{x - \bar x_0}{s_0},\ -c,\ c\right), \qquad c = 10.$$

The clip keeps a saturated valve from pulling every distance after it.

**Write**

- `per_fault_mi`: a `pd.DataFrame` indexed by `VARS` with the columns 1, 2, 4, 6, 7: your
  `binned_mi` of each variable with `fault` on the rows where `fault` is 0 or that fault
  (10 bins);
- `FIVE`: a list of five names from `VARS` that obey the two rules;
- `scale_to_normal(X, X0, clip=10)`: the rule above for 2-D arrays or DataFrames `X` (the data)
  and `X0` (the normal rows), column by column; returns a float `np.ndarray` of the shape of `X`;
- `ari_five`: a dict with the ARI against `y` of K-means on `tep[FIVE]`, with $k$ from 2 to 9
  chosen by the largest `silhouette_score` and `KMeans(k, n_init=10, random_state=0)` for both
  the search and the final fit: `"standard"` for `StandardScaler`, `"normal"` for
  `scale_to_normal` against the rows of fault 0.

In [ ]:
per_fault_mi = ...
FIVE = ...


def scale_to_normal(X, X0, clip=10):
    ...


ari_five = {"standard": ..., "normal": ...}

In [ ]:
# Tests for Task 7 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_scale_and_clip_by_hand():
    X0 = np.array([[0.0, 10.0], [2.0, 30.0]])
    got = scale_to_normal(np.array([[1.0, 20.0], [100.0, 0.0]]), X0)
    assert isinstance(got, np.ndarray) and got.shape == (2, 2), "an array of the shape of X"
    s = np.sqrt(2)
    assert np.allclose(got, [[0, 0], [10, -20 / (10 * s)]]), \
        (f"mean 1 and 20, std sqrt(2) and 10 sqrt(2); 99/sqrt(2) is clipped to 10; "
         f"got {got}")
    clipped = scale_to_normal(np.array([[100.0, 20.0]]), X0, clip=3)
    assert np.allclose(clipped, [[3, 0]]), "clip sets the limit"


test_scale_and_clip_by_hand()


def test_results_have_the_right_type():
    message = "per_fault_mi: VARS x faults 1, 2, 4, 6, 7"
    assert isinstance(per_fault_mi, pd.DataFrame), message
    assert list(per_fault_mi.columns) == [1, 2, 4, 6, 7], message
    assert set(per_fault_mi.index) == set(VARS), message
    assert isinstance(FIVE, list) and len(set(FIVE)) == 5 and set(FIVE) <= set(VARS), \
        "FIVE: five different variables"
    assert set(ari_five) == {"standard", "normal"}
    assert all(isinstance(v, float) for v in ari_five.values())


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_2_answers.md`, Task 7, with the numbers your code printed):

1. Print the best three variables of `per_fault_mi` for each fault. Explain the best one of each fault physically.
2. Write one line per variable of `FIVE`: what it is, which fault it covers, why you kept it over its neighbours. Report the largest $|r|$ inside the list.
3. Report `ari_five` with the $k$ the silhouette chose and the crosstabs, and compare with `ari_k6` and the DBSCAN of Task 4. Which fault still hides inside normal operation after standard scaling?
4. Why does it hide? Compare the standard deviation of your five over all data and within fault 0. What did `scale_to_normal` change, and if a fault is still merged, what are the other clusters spent on? Try one more $k$ or DBSCAN and report the ARI.
5. Which five sensors would you keep on this plant, and what do you lose by keeping only them?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

### Sources

- Downs J.J., Vogel E.F. A plant-wide industrial process control problem. *Computers & Chemical
  Engineering* 17(3), 1993. (the process, the variables, the faults)
- Braatz R.D. et al. Tennessee Eastman problem simulation data, the data set of Chiang, Russell
  and Braatz, *Fault Detection and Diagnosis in Industrial Systems*, Springer, 2001.
- Rousseeuw P.J. Silhouettes. *Journal of Computational and Applied Mathematics* 20, 1987.
- Ester M. et al. A density-based algorithm for discovering clusters. *KDD*, 1996. (DBSCAN)
- Hubert L., Arabie P. Comparing partitions. *Journal of Classification* 2, 1985. (ARI)
- Kraskov A., Stögbauer H., Grassberger P. Estimating mutual information. *Physical Review E*
  69, 2004. (the nearest-neighbour estimator)